# Лабораторная №4.
Получить в продолжении «психологическое», затем «опосредованным».
Промпт не содержит этих слов и их словоформ. Принудительное включение слов не используется.

Первый этап: 0 успешных продолжений из 72 запусков (P1–P6).

Второй этап: 3 промпта × 3 температуры × 20 seed = 180 продолжений.

Третий этап: 1 промт и тот же перебор

## 1. Среда и сохранение
В Colab выбрать GPU. Результаты записываются на Google Drive после каждой генерации.
При повторном выполнении готовые опыты пропускаются.

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_ROOT = Path('/content/huggingface')
    OUTPUT_DIR = Path('/content/drive/MyDrive/lab4_nlp/stage2')
else:
    CACHE_ROOT = Path(r'F:\huggingface') if Path('F:/').exists() else Path.home() / '.cache/huggingface'
    OUTPUT_DIR = Path('results/stage2')

os.environ['HF_HOME'] = str(CACHE_ROOT)
os.environ['HF_HUB_CACHE'] = str(CACHE_ROOT / 'hub')
os.environ['HF_XET_CACHE'] = str(CACHE_ROOT / 'xet')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import json
import re
import time
import hashlib
from collections import defaultdict
from html import escape

import torch
import transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GenerationConfig, set_seed
from IPython.display import display, HTML

MODEL_NAME = 'ai-forever/rugpt3large_based_on_gpt2'
TARGETS = ('психологическое', 'опосредованным')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_PATH = OUTPUT_DIR / 'generations.jsonl'
print('Устройство:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| Transformers:', transformers.__version__)
print('Результаты:', RESULTS_PATH)

Mounted at /content/drive
Устройство: cuda
Tesla T4
PyTorch: 2.11.0+cu130 | Transformers: 5.20.2
Результаты: /content/drive/MyDrive/lab4_nlp/stage2/generations.jsonl


## 2. Промпты и проверка


In [2]:
PROMPTS = {'P10': 'Сравним воздействие на тело без посредника и влияние на сознание через посредника.\n '
       'Физика — физическое воздействие, которое является непосредственным.\n'
       'Психология —'}

def check_prompt(text):
    words = re.findall(r'\w+', text.casefold())
    if any(w.startswith(('психологическ', 'опосредован')) for w in words):
        raise ValueError('В промпте найдена запрещённая словоформа.')

def analyze(text):
    matches = list(re.finditer(r'\w+', text.casefold()))
    positions = [[i for i, m in enumerate(matches) if m.group() == target]
                 for target in TARGETS]
    first, second = positions
    ordered = [(i, j) for i in first for j in second if i < j]
    snippets = []
    for match in matches:
        if match.group() in TARGETS:
            snippets.append(text[max(0, match.start() - 100):match.end() + 100])
    return {
        'first_found': bool(first),
        'second_found': bool(second),
        'both_found': bool(first and second),
        'success': bool(ordered),
        'positions': dict(zip(TARGETS, positions)),
        'other_forms': sorted({m.group() for m in matches
            if m.group().startswith(('психологическ', 'опосредован'))
            and m.group() not in TARGETS}),
        'snippets': snippets,
    }

for name, prompt in PROMPTS.items():
    check_prompt(prompt)
    print(name, prompt, '\n')

P10 Сравним воздействие на тело без посредника и влияние на сознание через посредника.
 Физика — физическое воздействие, которое является непосредственным.
Психология — 



## 3. Модель

In [3]:
tok = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE)
model.eval()
print('Модель загружена.')

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.14GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена.


## 4. Генерация и журнал
Проверяется только новое продолжение. Пользовательский `LogitsProcessor`,
принудительные слова, штрафы за повторы и запрет n-грамм не задаются.
Дополнительно фиксируются длина, причина остановки и доля повторных триграмм слов.

In [4]:
def repeat_fraction(text):
    words = re.findall(r'\w+', text.casefold())
    grams = list(zip(words, words[1:], words[2:]))
    return round(1 - len(set(grams)) / len(grams), 4) if grams else 0.0

def generate(prompt, temperature, top_k, top_p, max_new_tokens):
    check_prompt(prompt)
    inputs = tok(prompt, return_tensors='pt').to(DEVICE)
    prompt_length = inputs['input_ids'].shape[1]
    if prompt_length + max_new_tokens > model.config.max_position_embeddings:
        raise ValueError('Превышена длина контекста модели.')
    config = GenerationConfig(
        do_sample=True, num_beams=1,
        temperature=temperature, top_k=top_k, top_p=top_p,
        max_new_tokens=max_new_tokens,
        repetition_penalty=1.0, no_repeat_ngram_size=0,
        bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, generation_config=config)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    ids = output[0, prompt_length:].tolist()
    text = tok.decode(ids, skip_special_tokens=True)
    ended = bool(ids and ids[-1] == tok.eos_token_id)
    return {
        'continuation': text, 'seconds': round(elapsed, 3),
        'generated_tokens': len(ids), 'stop_reason': 'eos' if ended else 'length',
        'repeat_trigram_fraction': repeat_fraction(text), **analyze(text),
    }

def load_saved():
    saved = {}
    if RESULTS_PATH.exists():
        for number, line in enumerate(RESULTS_PATH.read_text(encoding='utf-8').splitlines(), 1):
            if line.strip():
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as error:
                    raise ValueError(f'Повреждена строка {number} журнала; сохраните копию файла перед исправлением.') from error
                saved[row['id']] = row
    return saved

def run_experiments(plan):
    saved = load_saved()
    rows = []
    for index, item in enumerate(plan, 1):
        spec = dict(item,
            model=MODEL_NAME, revision=getattr(model.config, '_commit_hash', None),
            targets=list(TARGETS), torch_version=str(torch.__version__),
            transformers_version=transformers.__version__, device=DEVICE,
            dtype=str(model.dtype), repetition_penalty=1.0, no_repeat_ngram_size=0,
        )
        key = hashlib.sha256(json.dumps(spec, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
        reused = key in saved
        if reused:
            row = saved[key]
        else:
            set_seed(spec['seed'])
            result = generate(spec['prompt'], spec['temperature'], spec['top_k'],
                              spec['top_p'], spec['max_new_tokens'])
            row = dict(spec, id=key, **result)
            with RESULTS_PATH.open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(row, ensure_ascii=False) + '\n')
            saved[key] = row
            if row['success']:
                (OUTPUT_DIR / f'success_{key[:12]}.json').write_text(
                    json.dumps(row, ensure_ascii=False, indent=2), encoding='utf-8')
        rows.append(row)
        print(f"{index}/{len(plan)} | {row['prompt_name']} | T={row['temperature']} | "
              f"seed={row['seed']} | успех={row['success']} | "
              f"токенов={row['generated_tokens']}" + (' | из файла' if reused else ''), flush=True)
    return rows

def show_table(headers, rows):
    head = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row)
                   + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

def summarize(rows):
    groups = defaultdict(list)
    for row in rows:
        key = (row['prompt_name'], row['temperature'], row['top_k'], row['top_p'], row['max_new_tokens'])
        groups[key].append(row)
    table = []
    for key, group in groups.items():
        n = len(group)
        table.append([*key, n,
            sum(r['first_found'] for r in group), sum(r['second_found'] for r in group),
            sum(r['both_found'] for r in group), sum(r['success'] for r in group),
            round(sum(r['generated_tokens'] for r in group) / n, 1),
            round(sum(r['repeat_trigram_fraction'] for r in group) / n, 3),
            round(sum(r['seconds'] for r in group) / n, 2)])
    show_table(['Промпт', 'T', 'top_k', 'top_p', 'Лимит', 'N', TARGETS[0], TARGETS[1],
                'Оба слова', 'По порядку', 'Токенов, ср.', 'Повторы, ср.', 'Секунд, ср.'], table)

## 5. Основной эксперимент
Меняется только температура; длина и фильтры фиксированы. Перебор завершается полностью,
чтобы сравнить конфигурации при одинаковом числе запусков.

In [5]:
TEMPERATURES = [0.8, 1.0, 1.2]
SEARCH_SEEDS = list(range(100, 120))
plan = [
    dict(stage='temperature', prompt_name=name, prompt=prompt,
         temperature=temperature, top_k=50, top_p=0.95,
         max_new_tokens=250, seed=seed)
    for name, prompt in PROMPTS.items()
    for temperature in TEMPERATURES
    for seed in SEARCH_SEEDS
]
print('Запланировано:', len(plan))
results = run_experiments(plan)

Запланировано: 60


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


1/60 | P10 | T=0.8 | seed=100 | успех=False | токенов=250
2/60 | P10 | T=0.8 | seed=101 | успех=False | токенов=250
3/60 | P10 | T=0.8 | seed=102 | успех=False | токенов=250
4/60 | P10 | T=0.8 | seed=103 | успех=False | токенов=250
5/60 | P10 | T=0.8 | seed=104 | успех=False | токенов=250
6/60 | P10 | T=0.8 | seed=105 | успех=False | токенов=250
7/60 | P10 | T=0.8 | seed=106 | успех=False | токенов=250
8/60 | P10 | T=0.8 | seed=107 | успех=False | токенов=250
9/60 | P10 | T=0.8 | seed=108 | успех=False | токенов=250
10/60 | P10 | T=0.8 | seed=109 | успех=False | токенов=250
11/60 | P10 | T=0.8 | seed=110 | успех=False | токенов=250
12/60 | P10 | T=0.8 | seed=111 | успех=False | токенов=250
13/60 | P10 | T=0.8 | seed=112 | успех=True | токенов=250
14/60 | P10 | T=0.8 | seed=113 | успех=False | токенов=250
15/60 | P10 | T=0.8 | seed=114 | успех=False | токенов=250
16/60 | P10 | T=0.8 | seed=115 | успех=False | токенов=250
17/60 | P10 | T=0.8 | seed=116 | успех=False | токенов=250
18/60 |

In [6]:
summarize(results)
print('Успешных продолжений:', sum(row['success'] for row in results), '/', len(results))

Промпт,T,top_k,top_p,Лимит,N,психологическое,опосредованным,Оба слова,По порядку,"Токенов, ср.","Повторы, ср.","Секунд, ср."
P10,0.8,50,0.95,250,20,7,4,2,2,250.0,0.391,4.87
P10,1.0,50,0.95,250,20,4,2,0,0,250.0,0.106,4.25
P10,1.2,50,0.95,250,20,2,0,0,0,250.0,0.008,4.34


Успешных продолжений: 2 / 60


При T=0.8 успех получен в 2 из 20 запусков (seed 112 и 117). При T=1.0 и 1.2 — 0 из 20.

## 6. Продолжения и контекст найденных слов

In [7]:
for row in sorted(results, key=lambda r: (r['success'], r['both_found'], r['second_found'], r['first_found']), reverse=True):
    title = (f"{row['prompt_name']} | T={row['temperature']} | seed={row['seed']} | "
             f"успех={row['success']} | формы: {', '.join(row['other_forms']) or '—'}")
    snippets = '\n…\n'.join(row['snippets']) or 'Точные словоформы не найдены.'
    body = ('Позиции слов (с нуля): ' + str(row['positions']) + '\n\n'
            + snippets + '\n\nПолное продолжение:\n' + row['continuation'])
    display(HTML('<details><summary>' + escape(title) + '</summary><pre style="white-space:pre-wrap">'
                 + escape(body) + '</pre></details>'))

## 7. Итог

In [9]:
all_results = results
successes = [row for row in all_results if row['success']]
print(f'Выполнено запусков: {len(all_results)}. Успешных: {len(successes)}.')
if successes:
    example = successes[0]
    print('Промпт:', example['prompt'])
    print('Параметры:', {k: example[k] for k in
          ['temperature', 'top_k', 'top_p', 'max_new_tokens', 'seed']})
    print('Продолжение:\n' + example['continuation'])
else:
    print('В проверенных конфигурациях условие не выполнено.')
print('Результаты и успешные примеры сохранены в', OUTPUT_DIR)

Выполнено запусков: 60. Успешных: 2.
Промпт: Сравним воздействие на тело без посредника и влияние на сознание через посредника.
 Физика — физическое воздействие, которое является непосредственным.
Психология —
Параметры: {'temperature': 0.8, 'top_k': 50, 'top_p': 0.95, 'max_new_tokens': 250, 'seed': 112}
Продолжение:
 психологическое воздействие, которое является опосредованным.
Магия — магическое воздействие, которое является опосредованным.
Бизнес — бизнес-попытка воздействовать на сознательность другого человека посредством его имени и/или образа.
Магия, колдовство — любые действия, направленные на изменение сознания человека и/или на изменение его поведения.
Влияние — влияние на психику или поведение другого человека, которое может быть направлено на достижение его собственных целей.
Цивилизация — цивилизация, которая имеет множество народов, культур и языков.
Судьба — совокупность обстоятельств, которые управляют человеческой жизнью и определяют её направление.
Культура — это сист

| Запуск | do_sample | seed | Температура | top_k / top_p | num_beams | Новых токенов | Результат |
|---|---|---:|---:|---|---:|---:|---|
| lab4.py, CPU float32 | False | Не влияет | Не используется | Не используются | 1 | 64 | Успех |
| P10, Colab GPU float16 | True | 112 | 0.8 | 50 / 0.95 | 1 | 250 | Успех |
| P10, Colab GPU float16 | True | 117 | 0.8 | 50 / 0.95 | 1 | 250 | Успех |